# A.4: Loading GPT-2's vocabulary and merges

Convert GPT-2's published files into the four tables and check that merge IDs grow with rank.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 The complete tokenizer (`bpe.py`)
import regex as re

from collections import Counter

# GPT-2's pre-tokenization pattern (Radford et al. 2019).
GPT2_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in the tuple `ids` with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)

class ByteBPE:
    def __init__(self, pattern=GPT2_PATTERN):
        self.pattern = re.compile(pattern)
        self.merges = {}                               # (id, id) -> new id
        self.vocab = {i: bytes([i]) for i in range(256)}  # id -> bytes
        self.special = {}                              # str -> id
        self.byte_ids = list(range(256))               # byte value -> id

    # ---- training: Steps 1 and 2 ---------------------------------------
    def train(self, text, vocab_size):
        chunks = Counter(self.pattern.findall(text))
        words = {tuple(c.encode("utf-8")): n for c, n in chunks.items()}
        for new_id in range(256, vocab_size):
            stats = Counter()
            for w, n in words.items():
                for pair in zip(w, w[1:]):
                    stats[pair] += n
            if not stats:
                break
            pair = max(stats, key=stats.get)   # ties: first pair seen
            words = {merge(w, pair, new_id): n for w, n in words.items()}
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]

    # ---- encoding: Step 3 ----------------------------------------------
    def _encode_chunk(self, ids):
        while len(ids) >= 2:
            # Merge the pair that was learned earliest. Merge IDs grow with
            # rank, so the pair with the smallest merge ID has the lowest rank.
            pair = min(zip(ids, ids[1:]),
                       key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return list(ids)

    def encode_ordinary(self, text):
        ids = []
        for chunk in self.pattern.findall(text):
            ids.extend(self._encode_chunk(
                tuple(self.byte_ids[b] for b in chunk.encode("utf-8"))))
        return ids

    def encode(self, text, allow_special=False):
        if not (allow_special and self.special):
            return self.encode_ordinary(text)
        split = "(" + "|".join(re.escape(s) for s in self.special) + ")"
        ids = []
        for part in re.split(split, text):
            if part in self.special:
                ids.append(self.special[part])
            elif part:
                ids.extend(self.encode_ordinary(part))
        return ids

    # ---- special tokens and decoding: Steps 4 and 5 --------------------
    def add_special_tokens(self, tokens):
        for t in tokens:
            if t not in self.special:
                new_id = len(self.vocab)
                self.special[t] = new_id
                self.vocab[new_id] = t.encode("utf-8")

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8", errors="replace")


In [ ]:
import json


def bytes_to_unicode():
    """GPT-2's reversible map from the 256 byte values to printable characters."""
    bs = (list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))


def load_gpt2(tok, vocab_path, merges_path):
    """Load GPT-2's published vocab.json and merges.txt into a ByteBPE."""
    byte_decoder = {c: b for b, c in bytes_to_unicode().items()}
    to_bytes = lambda s: bytes(byte_decoder[c] for c in s)
    with open(vocab_path, encoding="utf-8") as f:
        encoder = json.load(f)                        # token string -> id
    tok.vocab = {i: to_bytes(s) for s, i in encoder.items()}
    token_to_id = {b: i for i, b in tok.vocab.items()}
    tok.byte_ids = [token_to_id[bytes([b])] for b in range(256)]
    tok.merges = {}
    with open(merges_path, encoding="utf-8") as f:
        lines = f.read().split("\n")[1:]              # skip the version line
    for line in lines:
        if not line:
            continue
        a, b = line.split(" ")
        a, b = to_bytes(a), to_bytes(b)
        tok.merges[(token_to_id[a], token_to_id[b])] = token_to_id[a + b]
    # The merge dict maps pairs to ids; GPT-2's ids grow with merge rank,
    # so "lowest id" and "earliest merge" pick the same pair.
    tok.special = {"<|endoftext|>": encoder["<|endoftext|>"]}
    return tok


In [ ]:
from huggingface_hub import hf_hub_download

vocab_path = hf_hub_download("openai-community/gpt2", "vocab.json")
merges_path = hf_hub_download("openai-community/gpt2", "merges.txt")
gpt2_tok = load_gpt2(ByteBPE(), vocab_path, merges_path)

with open(merges_path, encoding="utf-8") as f:
    ranked = [line for line in f.read().split("\n")[1:] if line]
ok = all(gpt2_tok.merges[pair] == 256 + r
         for r, pair in enumerate(gpt2_tok.merges))
print(len(ranked), len(gpt2_tok.merges), len(gpt2_tok.vocab), ok)


**Expected output**

Output:

```text
50000 50000 50257 True
```
